# Multi-qubit NMR PiQC

Abstract solver instantiated with the MATLAB NMR Hamiltonians (`H0` from ZZ couplings, local `X,Y` drives) for 2- and 4-qubit molecules. Paper-scale: `T=8.8`, `n_traj=400`, `n_steps=100`, `n_pulses=50`, `n_iterations=1000`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from piqcx import AnnealingConfig, PiQC, SmoothingConfig, configure, devices
from piqcx.systems import nmr_params, nmr_problem

configure(precision="float32")
print("devices:", devices())

## Two-qubit molecule

In [ ]:
nmr2 = nmr_problem(2, time=1.0, diffusion=5e-4, control_weight=1.0, beta=40.0)
print("dim", nmr2.dim, "n_c", nmr2.n_controls, "R", nmr2.control_cost, "Q", nmr2.terminal_weight)
print("params\n", nmr_params(2))

res2 = PiQC(
    nmr2,
    n_traj=64,
    n_steps=40,
    n_pulses=20,
    n_iterations=25,
    annealing=AnnealingConfig(enabled=True, schedule="exponential", d_init=1e-3, d_final=1e-8),
    smoothing=SmoothingConfig(kind="window", window=1, window_after=6, window_late=6),
    seed=0,
).run()
print("F", float(res2.fidelity[-1]))

In [ ]:
t = np.linspace(res2.dt, res2.time, res2.n_steps)
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
axes[0, 0].plot(res2.fidelity)
axes[0, 0].set_ylabel("F")
axes[0, 1].semilogy(1.0 - np.clip(res2.fidelity, 0, 1 - 1e-12))
axes[0, 1].set_ylabel("1-F")
axes[1, 0].plot(t, res2.controls.T)
axes[1, 0].set_ylabel("u(t)")
axes[1, 1].semilogy(res2.diffusion_schedule)
axes[1, 1].set_ylabel("D")
fig.tight_layout()

## Four-qubit molecule (Hamiltonian + short smoke run)

The 4-qubit Hilbert space is 16-dimensional (`n_c=8`). Increase `n_traj` / `n_iterations` for production.

In [ ]:
nmr4 = nmr_problem(4, time=1.0, diffusion=1e-4, control_weight=1.0, beta=20.0)
print("dim", nmr4.dim, "||H0||", float(np.linalg.norm(np.asarray(nmr4.h0))))

res4 = PiQC(
    nmr4,
    n_traj=32,
    n_steps=24,
    n_pulses=8,
    n_iterations=8,
    annealing=AnnealingConfig(enabled=True, schedule="steps", d_init=5e-4, d_final=1e-7, n_plateaus=4),
    seed=2,
).run()
print("F", float(res4.fidelity[-1]), "ESS", float(res4.ess[-1]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(res4.fidelity)
axes[0].set_title("4-qubit fidelity")
for q in range(nmr4.n_qubits):
    axes[1].plot(res4.controls[2 * q], label=f"qx{q}")
axes[1].set_title("X pulses")
axes[1].legend(fontsize=8)
fig.tight_layout()